In [7]:
import os
import re
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from keras.models import Model, Sequential
from keras.layers import (Input, LSTM, Dense, Dot, Activation, Concatenate, Dropout,
                          Flatten, Permute, Layer, GlobalMaxPooling2D, Embedding,
                          Bidirectional, TimeDistributed, Lambda, BatchNormalization)
from keras.callbacks import EarlyStopping
from keras.regularizers import l2
from keras.optimizers import Adam
from keras.preprocessing.sequence import pad_sequences
from hazm import *
from gensim.models import Word2Vec

In [8]:
normalizer = Normalizer()
stemmer = Stemmer()
lemmatizer = Lemmatizer()

In [9]:
# Pre-compile regex patterns for efficiency
website_pattern = re.compile(r'https?://\S+|www\.\S+')
email_pattern = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
number_pattern = re.compile(r'[0-9۰-۹]+')
english_pattern = re.compile(r'[a-zA-Z]+')
repeated_char_pattern = re.compile(r'(.)\1+')
english_punctuation = string.punctuation
persian_punctuation = "،؛؟«»"
all_punctuation = english_punctuation + persian_punctuation
unnecessary_chars = set(all_punctuation)

def clean_text(text):
    text = str(text).replace('\u200c', ' ')
    text = re.sub(website_pattern, ' ', text)
    text = re.sub(email_pattern, ' ', text)
    text = re.sub(number_pattern, ' ', text)
    text = re.sub(english_pattern, ' ', text)
    text = re.sub(repeated_char_pattern, r'\1', text)
    text = ''.join(char if char.isalnum() or char.isspace() else ' ' for char in text)
    text = ' '.join(text.split())
    return text


def normalization(text):
    normalized_text = normalizer.normalize(text)
    return normalized_text

def tokenization(text):
    tokens = word_tokenize(text)
    return tokens

def get_stop_words(self):
    with open("stop_words.txt", 'r', encoding='utf-8') as file:
        stop_words = set(word.strip() for word in file.readlines())
    return stop_words

def handle_missed():
    dataset['abstract'] = dataset['abstract'].fillna('')
    dataset['body'] = dataset['body'].fillna('')


def find_stop_words(tfidf_vectorizer):
    idf_scores = tfidf_vectorizer.idf_
    feature_names = tfidf_vectorizer.get_feature_names_out()
    max_idf_threshold = 11
    min_idf_threshold = 2
    stop_words = [word for word, score in zip(feature_names, idf_scores) if not (min_idf_threshold <=score <= max_idf_threshold)]
    print(len(stop_words))
    with open("stop_words.txt", 'w', encoding='utf-8') as file:
        for word in stop_words:
            file.write(word + '\n')
    print("Stop words saved to: stop_words.txt")

def get_stop_words():
        with open("stop_words.txt", 'r', encoding='utf-8') as file:
            stop_words = set(word.strip() for word in file.readlines())
        return stop_words

def remove_stop_words(tokens, stop_words):
    filtered_tokens = [token for token in tokens if token not in stop_words]
    return filtered_tokens

def remove_unnecessary_characters(words):
    filtered_tokens = [token for token in words if token not in unnecessary_chars]
    return filtered_tokens

def stemming(tokens):
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens


def lemmatizing(tokens):
    lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]

    def handle_special_token(tokens):
        filtered_tokens = []
        for token in tokens:
            parts = token.split('#')
            if len(parts) == 1:
                filtered_tokens.append(token)
                continue
            filtered_tokens.append(parts[1])
        return filtered_tokens

    lemmatized_words = handle_special_token(lemmatized_words)
    return lemmatized_words

In [3]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        
    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        e = tf.keras.activations.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

In [4]:

model = load_model('model.keras', custom_objects={'AttentionLayer': AttentionLayer})


In [10]:
file_path = "news_data.csv"
dataset = pd.read_csv(file_path)